# Notebook 19 — Activation Patching Localization of Sequential Edit Damage
**CS 590NN | Amogh | Apr 25 — bridging interpretability and editing**

## Genuinely novel angle (verified against literature)

Activation patching (Vig 2020, Meng ROME 2022, Wang IOI 2022) is a standard interpretability technique: replace activations from a "clean" run with activations from a "corrupted" run to localize where a computation lives.

**No paper applies activation patching to localize the damage from a sequential edit.** This combines:
- Activation patching methodology (interpretability literature)
- Sequential edit damage (editing literature)
- Bistability characterization from NB16 (your novel finding)

The specific question — *at which layer does the bistable collapse occur?* — is unasked in either literature.

## Protocol

For each pair (A, B):

1. **Cache phase:** Edit A → forward A's prompt → cache `acts_postA[L]` (residual stream at every layer L)
2. **Damage phase:** Restore. Edit A then B → confirm A is destroyed
3. **Single-layer patching:** For each layer L in 0..n_layers-1: run forward on A's prompt with current weights, but inject `acts_postA[L]` at layer L. Measure recovered p_new_A.
4. **Cumulative patching:** Patch layers 0..L all at once. Measure where recovery saturates.

## Predictions

| Outcome | Interpretation |
|---|---|
| Single layer L\* recovers A fully | **Damage is local at L\*.** Compare to A's ROME-trace MLPs. |
| Cumulative needs many layers, no single one suffices | **Damage is distributed.** Restoration requires fixing the residual stream at multiple sites. |
| No patching recovers A | **Damage saturates the residual stream.** Connects to NB16's "damage faster than installation." |
| Damage concentrated in layers B never modified | **Damage cascades downstream of edit site.** Strongest novelty — overturns intuition that damage is at modified parameters. |

Each outcome is publishable on its own merits.

## Compute

~15-20 min on A100. ~5600 forward passes total (very fast each).

### 19.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 19.1 Imports + auto-fetch v3

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb19'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 8: break
print(f'Selected {len(PAIR_INDICES)} pairs')

### 19.2 Load model + samples

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token

@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Loaded {len(pairs)} pairs, n_layers={model.cfg.n_layers}')

### 19.3 ROME-trace + edit helpers

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def edit_normal(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Helpers ready.')

### 19.4 Activation caching and patching

In [ ]:
def cache_residual_per_layer(model, sample):
    """Cache residual stream at every layer (post-block) for the last token position."""
    model.eval()
    tokens = model.to_tokens(sample.prompt)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    acts = {}
    for L in range(model.cfg.n_layers):
        h_name = f'blocks.{L}.hook_resid_post'
        if h_name not in cache:
            h_name = f'blocks.{L}.hook_resid_pre'
        acts[L] = cache[h_name][:, :, :].detach().clone()  # full residual: (1, n_tok, d_model)
    del cache; torch.cuda.empty_cache()
    return acts, tokens

def patch_single_layer(model, tokens, acts_clean, layer_to_patch, sample):
    """
    Run forward on `tokens` with current model, but at `layer_to_patch`
    replace the residual stream with `acts_clean[layer_to_patch]`.
    Return p_new for `sample.target_new`.
    """
    model.eval()
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    
    h_name_post = f'blocks.{layer_to_patch}.hook_resid_post'
    h_name_pre = f'blocks.{layer_to_patch}.hook_resid_pre'
    # Determine which hook name actually exists in this model
    test_cache = None
    
    def patch_hook(value, hook):
        return acts_clean[layer_to_patch]
    
    # Try post first, fall back to pre
    try:
        with torch.no_grad():
            logits = model.run_with_hooks(tokens, fwd_hooks=[(h_name_post, patch_hook)])
    except (KeyError, RuntimeError):
        with torch.no_grad():
            logits = model.run_with_hooks(tokens, fwd_hooks=[(h_name_pre, patch_hook)])
    
    p_new = float(torch.softmax(logits[0,-1,:], dim=-1)[new_id])
    return p_new

def patch_cumulative_layers(model, tokens, acts_clean, layers_to_patch, sample):
    """Patch all layers in layers_to_patch simultaneously."""
    model.eval()
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    
    hooks = []
    for L in layers_to_patch:
        h_name = f'blocks.{L}.hook_resid_post'
        # closure over L
        def make_hook(L=L):
            def patch_hook(value, hook):
                return acts_clean[L]
            return patch_hook
        hooks.append((h_name, make_hook()))
    
    try:
        with torch.no_grad():
            logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
    except (KeyError, RuntimeError):
        # try resid_pre
        hooks_pre = []
        for L in layers_to_patch:
            h_name = f'blocks.{L}.hook_resid_pre'
            def make_hook(L=L):
                def patch_hook(value, hook):
                    return acts_clean[L]
                return patch_hook
            hooks_pre.append((h_name, make_hook()))
        with torch.no_grad():
            logits = model.run_with_hooks(tokens, fwd_hooks=hooks_pre)
    
    p_new = float(torch.softmax(logits[0,-1,:], dim=-1)[new_id])
    return p_new

print('Patching utilities ready.')

### 19.5 Run the full protocol

In [ ]:
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR = 5e-3

ref_cache = cache_ref(model, NEUTRAL)
all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'ROME-trace for {len(all_samples)} samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

print('Snapshotting original weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()

ROWS_FILE = RESULTS_DIR / f'patching_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
all_results = []
started = datetime.now()

for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    layers_A = circuits[A.idx]
    layers_B = circuits[B.idx]
    
    try:
        # === Phase 1: Cache acts after edit A ===
        restore(model, original_state)
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        p_A_postA = measure_p_new(model, A)
        acts_postA, tokens_A = cache_residual_per_layer(model, A)
        
        # === Phase 2: Edit A then B (sequential), confirm A destroyed ===
        restore(model, original_state)
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        edit_normal(model, B, layers_B, n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        p_A_postAB = measure_p_new(model, A)
        
        if p_A_postAB > 0.5:
            print(f'[{p_idx+1}/{len(pairs)}] pair {p_idx}: A NOT destroyed (p_A_postAB={p_A_postAB:.3f}), skipping')
            continue
        
        recovery_range = max(p_A_postA - p_A_postAB, 0.01)  # avoid div by 0
        
        # === Phase 3: Single-layer patching ===
        single_layer_recovery = []
        for L in range(model.cfg.n_layers):
            p_recovered = patch_single_layer(model, tokens_A, acts_postA, L, A)
            recovery_ratio = (p_recovered - p_A_postAB) / recovery_range
            single_layer_recovery.append({'layer': L, 'p_recovered': p_recovered, 'recovery_ratio': recovery_ratio})
        
        # === Phase 4: Cumulative patching (layers 0..L) ===
        cumulative_recovery = []
        for L in range(model.cfg.n_layers):
            layers_to_patch = list(range(L+1))
            p_recovered = patch_cumulative_layers(model, tokens_A, acts_postA, layers_to_patch, A)
            recovery_ratio = (p_recovered - p_A_postAB) / recovery_range
            cumulative_recovery.append({'up_to_layer': L, 'p_recovered': p_recovered, 'recovery_ratio': recovery_ratio})
        
        # Find peak single-layer recovery
        peak_single = max(single_layer_recovery, key=lambda x: x['recovery_ratio'])
        # Find layer where cumulative reaches 0.5 recovery
        cum_50 = next((x for x in cumulative_recovery if x['recovery_ratio'] >= 0.5), None)
        cum_50_layer = cum_50['up_to_layer'] if cum_50 else None
        
        result = {
            'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx,
            'layers_A_edit': layers_A, 'layers_B_edit': layers_B,
            'p_A_postA': p_A_postA, 'p_A_postAB': p_A_postAB,
            'recovery_range': recovery_range,
            'peak_single_layer': peak_single['layer'],
            'peak_single_recovery': peak_single['recovery_ratio'],
            'cumulative_50_layer': cum_50_layer,
            'single_layer_recovery': single_layer_recovery,
            'cumulative_recovery': cumulative_recovery,
            'status': 'ok',
        }
        all_results.append(result)
        print(f'[{p_idx+1}/{len(pairs)}] pair {p_idx}: A_postA={p_A_postA:.3f}→postAB={p_A_postAB:.3f}  '
              f'peak L={peak_single["layer"]} (rec={peak_single["recovery_ratio"]:.2f})  '
              f'cum 50% at L={cum_50_layer}')
        
        with open(ROWS_FILE, 'w') as f: json.dump({'results': all_results}, f, indent=2)
    except Exception as e:
        print(f'[{p_idx+1}/{len(pairs)}] FAILED: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 19.6 Analyze and characterize the localization

In [ ]:
ok_results = [r for r in all_results if r.get('status') == 'ok']
print(f'Successful pairs: {len(ok_results)}')

# Per-layer mean recovery across all pairs
n_layers = model.cfg.n_layers
single_per_layer = np.zeros((len(ok_results), n_layers))
cumul_per_layer = np.zeros((len(ok_results), n_layers))
for i, r in enumerate(ok_results):
    for entry in r['single_layer_recovery']:
        single_per_layer[i, entry['layer']] = entry['recovery_ratio']
    for entry in r['cumulative_recovery']:
        cumul_per_layer[i, entry['up_to_layer']] = entry['recovery_ratio']

mean_single = single_per_layer.mean(axis=0)
mean_cumul  = cumul_per_layer.mean(axis=0)

print('\n=== Mean single-layer patch recovery per layer ===')
peak_layer = int(np.argmax(mean_single))
print(f'  Peak layer: L{peak_layer}, mean recovery = {mean_single[peak_layer]:.3f}')
print(f'  Top 5 layers: {sorted(range(n_layers), key=lambda L: -mean_single[L])[:5]}')

print('\n=== Cumulative patch recovery ===')
cum_50_pct = next((L for L in range(n_layers) if mean_cumul[L] >= 0.5), None)
cum_90_pct = next((L for L in range(n_layers) if mean_cumul[L] >= 0.9), None)
print(f'  Reaches 50% recovery by patching layers 0..{cum_50_pct}')
print(f'  Reaches 90% recovery by patching layers 0..{cum_90_pct}')

# Critical question: does the peak single-layer recovery match A's edited MLPs?
print('\n=== Does damage localize at A\'s edit sites? ===')
match_count = 0
for r in ok_results:
    edit_layers = set(r['layers_A_edit'])
    peak_L = r['peak_single_layer']
    in_edit = peak_L in edit_layers
    nearby = any(abs(peak_L - el) <= 2 for el in edit_layers)
    print(f'  pair {r["pair_idx"]}: peak L={peak_L}, A edited at {sorted(edit_layers)}, '
          f'in edit set: {in_edit}, within 2: {nearby}')
    if in_edit: match_count += 1
print(f'\n  {match_count}/{len(ok_results)} pairs: peak recovery layer is in A\'s ROME-trace edit set.')

# Distribution: is damage localized or distributed?
print('\n=== Localization vs distribution ===')
localized = 0  # peak single-layer recovery > 0.5
distributed = 0  # peak single-layer < 0.3 but cumulative reaches 0.5
saturated = 0  # cumulative never reaches 0.5
for r in ok_results:
    peak = r['peak_single_recovery']
    cum_50 = r['cumulative_50_layer']
    if peak > 0.5:
        localized += 1
    elif cum_50 is not None:
        distributed += 1
    else:
        saturated += 1
print(f'  Localized (peak single-layer recovery > 0.5):     {localized}/{len(ok_results)}')
print(f'  Distributed (cumul ≥ 0.5 but no single layer):    {distributed}/{len(ok_results)}')
print(f'  Saturated (cumul never reaches 0.5):              {saturated}/{len(ok_results)}')

# Verdict
print('\n=== VERDICT ===')
if localized > distributed and localized > saturated:
    if match_count >= len(ok_results) * 0.5:
        print('  >>> NOVEL FINDING: Damage is LOCALIZED at A\'s ROME-trace edit sites.')
        print('  >>> Mechanism: optimizer step damages the same MLPs A used. Confirms intuition.')
    else:
        print('  >>> NOVEL FINDING: Damage is LOCALIZED but at layers DIFFERENT from A\'s edit sites.')
        print('  >>> Mechanism: damage cascades downstream. Strongest novelty.')
elif distributed > localized:
    print('  >>> NOVEL FINDING: Damage is DISTRIBUTED across multiple layers.')
    print('  >>> No single layer carries the damage. Restoration requires fixing many layers.')
elif saturated > 0:
    print(f'  >>> NOVEL FINDING: Damage SATURATES residual stream in {saturated}/{len(ok_results)} pairs.')
    print('  >>> Even patching all early layers fails to recover A. Connects to NB16 temporal asymmetry.')
else:
    print('  >>> Mixed result, see per-pair table above.')

### 19.7 Money plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left — single-layer recovery curves per pair
ax = axes[0]
for i, r in enumerate(ok_results):
    layer_xs = [e['layer'] for e in r['single_layer_recovery']]
    rec_ys = [e['recovery_ratio'] for e in r['single_layer_recovery']]
    ax.plot(layer_xs, rec_ys, alpha=0.4, lw=1.0, label=f'pair {r["pair_idx"]}')
ax.plot(range(n_layers), mean_single, 'k-', lw=3, label='MEAN', zorder=10)
ax.axhline(0.5, color='red', ls='--', alpha=0.5, label='50% recovery threshold')
ax.set_xlabel('Patched layer L')
ax.set_ylabel('Recovery ratio')
ax.set_title('Single-layer patching: which layer alone recovers A?')
ax.grid(alpha=0.3); ax.legend(loc='upper right', fontsize=7)

# Middle — cumulative recovery curves
ax = axes[1]
for i, r in enumerate(ok_results):
    layer_xs = [e['up_to_layer'] for e in r['cumulative_recovery']]
    rec_ys = [e['recovery_ratio'] for e in r['cumulative_recovery']]
    ax.plot(layer_xs, rec_ys, alpha=0.4, lw=1.0)
ax.plot(range(n_layers), mean_cumul, 'k-', lw=3, label='MEAN', zorder=10)
ax.axhline(0.5, color='red', ls='--', alpha=0.5, label='50%')
ax.axhline(0.9, color='orange', ls='--', alpha=0.5, label='90%')
ax.set_xlabel('Patch layers 0..L')
ax.set_ylabel('Cumulative recovery')
ax.set_title('Cumulative patching: how many layers needed?')
ax.grid(alpha=0.3); ax.legend()

# Right — mean recovery vs A's edit-site distribution
ax = axes[2]
ax.bar(range(n_layers), mean_single, color='steelblue', alpha=0.7, label='Mean single-L recovery')
# Overlay A's edit sites as vertical markers
all_edit_layers = []
for r in ok_results:
    all_edit_layers.extend(r['layers_A_edit'])
edit_freq = np.zeros(n_layers)
for L in all_edit_layers:
    edit_freq[L] += 1
edit_freq = edit_freq / max(edit_freq.max(), 1)
ax2 = ax.twinx()
ax2.bar(range(n_layers), edit_freq, color='red', alpha=0.4, label='A edit-site frequency')
ax2.set_ylabel('A edit-site frequency (normalized)', color='red')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean recovery ratio')
ax.set_title('Damage localization vs A\'s edit sites')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_patching_localization.png', dpi=140)
plt.show()

### 19.8 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

summary = {
    'notebook': 'Notebook 19 — Activation Patching Localization of Damage',
    'timestamp': ts, 'model': MODEL_NAME, 'n_pairs': len(ok_results),
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'mean_single_layer_recovery_per_layer': mean_single.tolist(),
    'mean_cumulative_recovery_per_layer': mean_cumul.tolist(),
    'peak_single_layer_overall': int(peak_layer),
    'peak_single_layer_mean_recovery': float(mean_single[peak_layer]),
    'cumulative_50pct_layer': int(cum_50_pct) if cum_50_pct is not None else None,
    'cumulative_90pct_layer': int(cum_90_pct) if cum_90_pct is not None else None,
    'pairs_with_localized_damage': localized,
    'pairs_with_distributed_damage': distributed,
    'pairs_with_saturated_damage': saturated,
    'damage_at_edit_site_count': match_count,
    'total_pairs': len(ok_results),
    'verdict': ('localized_at_edit_site' if (localized > 0 and match_count > localized * 0.5)
                else 'localized_downstream' if localized > 0
                else 'distributed' if distributed > saturated
                else 'saturated'),
}
with open(RESULTS_DIR / f'summary_nb19_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
# Save per-pair details too
df = pd.DataFrame([{
    'pair_idx': r['pair_idx'], 'A_idx': r['A_idx'], 'B_idx': r['B_idx'],
    'layers_A_edit': str(r['layers_A_edit']),
    'p_A_postA': r['p_A_postA'], 'p_A_postAB': r['p_A_postAB'],
    'peak_single_layer': r['peak_single_layer'],
    'peak_single_recovery': r['peak_single_recovery'],
    'cumulative_50_layer': r['cumulative_50_layer'],
} for r in ok_results])
df.to_csv(RESULTS_DIR / f'df_nb19_{ts}.csv', index=False)

print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb19_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')